Análisis de Resultados: Línea Base mediante Estadística Tradicional (Regresión Logística)
Para establecer un punto de comparación robusto (baseline) frente a los algoritmos de aprendizaje automático no lineal (como Random Forest y XGBoost), se implementó un modelo de Regresión Logística penalizada. Este enfoque representa la metodología de estadística tradicional frecuentemente utilizada en la epidemiología clásica.


1. Estabilidad y Convergencia del Modelo
El ajuste de hiperparámetros evaluó la fuerza de regularización y el tipo de penalidad (L1 Lasso vs. L2 Ridge). Se observó que, a pesar de otorgar 1.000 iteraciones máximas al solucionador saga (optimizado para grandes volúmenes de datos), el modelo presentó alertas de no convergencia en la predicción de Mortalidad. Este fenómeno es un hallazgo relevante, ya que demuestra la alta dimensionalidad y la naturaleza fuertemente no lineal de las interacciones en los datos oncológicos GRD, limitando la capacidad de los modelos lineales para separar óptimamente las clases.

No obstante, las configuraciones evaluadas demostraron ser altamente estables frente a la partición de datos, obteniendo desviaciones estándar nulas (std < 0.001) en la validación cruzada para las tres variables objetivo.

2. Evaluación Predictiva: El Límite de la Estadística Clásica
A) Mortalidad (Clasificación Binaria):
La Regresión Logística obtuvo un F1-Macro de 0.5791. Este resultado es de vital importancia metodológica, puesto que no logra superar el umbral mínimo de 0.60 establecido en los criterios de éxito del estudio. A pesar de contar con un excelente AUC-ROC (0.9424) influenciado por la alta tasa de verdaderos negativos, el modelo colapsa en la detección de la clase crítica (F1-Score Clase 1: 0.2362). Esto justifica empíricamente la necesidad de aplicar ensambles de árboles (como Random Forest, que alcanzó un F1-Macro de 0.68) para modelar este fenómeno.

B) Severidad y Consumo de Recursos (Clasificación Multiclase):
Para los enfoques de gestión hospitalaria, el modelo lineal logró un desempeño marginalmente superior al umbral mínimo, alcanzando un F1-Macro de 0.6889 en Severidad y 0.6714 en Consumo de Recursos. En ambos casos, las calibraciones de probabilidad resultaron excelentes (Brier Score de 0.0990 y 0.1439, respectivamente). Aunque válidos operativamente, estos modelos siguen estando por debajo del poder predictivo alcanzado por métodos de Machine Learning avanzado.

3. Interpretabilidad Directa (Odds Ratios Clínicos)
La principal ventaja metodológica de la Regresión Logística es la extracción directa de Odds Ratios (OR) desde sus coeficientes. El análisis del vector de pesos para la variable Mortalidad confirma la coherencia clínica de los datos GRD:

Asociaciones de Riesgo: Ingresar por la especialidad de Cuidados Críticos incrementa exponencialmente las chances base de fallecimiento (OR: 13.3), seguido por ingresos a través de la unidad de Urgencia Médica (OR: 6.5) y especialidades Respiratorias (OR: 6.2).

Asociaciones de Supervivencia: Los episodios asociados a procedimientos de Odontología, Oftalmología y control Psiquiátrico presentaron los Odds Ratios más bajos de la cohorte (acercándose a 0.0), validando que estos episodios carecen de riesgo vital inherente en el historial hospitalario público.

Conclusión Parcial:
La Regresión Logística logra extraer el sentido clínico direccional de las variables del sistema GRD. Sin embargo, su insuficiencia matemática para predecir la mortalidad (F1 < 0.60) ratifica formalmente la superioridad y necesidad técnica de los modelos de aprendizaje automático no lineal propuestos en esta investigación.

In [1]:
import pandas as pd # Librería para manipulación y análisis de datos tabulares en DataFrames
import numpy as np # Librería para cálculos numéricos avanzados y manejo de arreglos matemáticos
import os # Módulo para interactuar con el sistema operativo (creación de carpetas, manejo de rutas)
import gc # Recolector de basura (Garbage Collector) para liberar memoria RAM manualmente
import time # Módulo para registrar y medir el tiempo exacto de ejecución de los procesos
from sklearn.model_selection import GridSearchCV # Herramienta para la búsqueda exhaustiva de hiperparámetros
from sklearn.model_selection import StratifiedKFold # Importa la validación cruzada estratificada con semilla fija
from sklearn.linear_model import LogisticRegression # Importa el algoritmo estadístico base de Regresión Logística
from sklearn.preprocessing import StandardScaler, label_binarize # Escalador de variables y el transformador multiclase
from sklearn.pipeline import Pipeline # Importa el orquestador de pasos secuenciales para evitar fuga de datos al escalar
from sklearn.base import clone # Importa la función para clonar la arquitectura exacta de un modelo vacío
from sklearn.metrics import ( # Importa el bloque de herramientas para evaluar el rendimiento predictivo
    f1_score, # Métrica que evalúa el balance entre precisión y exhaustividad (recall)
    average_precision_score, # Métrica para calcular el Área Bajo la Curva Precision-Recall (AUPRC)
    roc_auc_score, # Métrica para calcular el Área Bajo la Curva ROC (AUC-ROC)
    brier_score_loss, # Métrica para evaluar qué tan bien calibradas están las probabilidades del modelo
    classification_report # Genera un reporte textual estructurado con precision, recall y f1 por cada clase
)

def entrenar_evaluar_lr(target_name):
    """
    - Descripción: Función que entrena, optimiza y evalúa un modelo de Regresión Logística mediante un Pipeline.
                   Aplica validación cruzada estratificada con semilla fija, escala las variables continuas internamente,
                   descarta hiperparámetros inestables (con desviación estándar > 0.10) y extrae Odds Ratios.
    - Entrada: target_name (str) - Nombre de la columna objetivo a predecir (ej. 'MORTALIDAD', 'SEVERIDAD').
    - Salida: Genera archivos CSV con el historial de GridSearch, métricas del entrenamiento y Odds Ratios.
    """
    # 1. Configuración inicial
    dir_datos = "../../Datos/Datasets Finales" # Ruta relativa donde se encuentran los archivos CSV de entrada
    dir_resultados = "../../Resultados/Resultados (etapa 3)/Regresion_Logistica" # Ruta donde se guardarán los resultados analíticos
    os.makedirs(dir_resultados, exist_ok=True) # Crea el directorio de resultados de forma segura si no existe

    # Lista de columnas que no deben usarse como predictores (incluyendo targets)
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60) 
    print(f"INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: {target_name.upper()}")
    print("="*60) 

    # 2. Cargar datos de entrenamiento
    print("[1/5] Cargando datasets de entrenamiento...") # Etapa 1: Carga de datos desde archivos CSV para ambos grupos (oncológicos y controles)
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco_2019_2023.csv"), low_memory=False) # Carga historial de pacientes oncológicos
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control_2019_2023.csv"), low_memory=False) # Carga historial del grupo control

    # 3. Crear Dataset Maestro Balanceado (Grupo oncológico vs control)
    print("[2/5] Generando muestra balanceada...") # Etapa 2: Balanceo de clases mediante submuestreo aleatorio del grupo control para igualar el número de pacientes oncológicos
    n_onco = len(df_onco_train) # Cuenta el número exacto de filas (pacientes) oncológicos en el entrenamiento
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) # Toma una muestra aleatoria de controles del mismo tamaño, fijando la semilla
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) # Apila ambos conjuntos verticalmente para formar el dataset final

    del df_onco_train, df_control_train, df_control_sample # Elimina de memoria las variables temporales gigantes
    gc.collect() # Invoca al recolector de basura de Python para liberar físicamente la memoria RAM

    # Separar X e Y de entrenamiento
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] # Filtra las columnas, reteniendo únicamente las variables independientes
    X_train = df_train_maestro[features] # Crea la matriz de características o predictores (X)
    y_train = df_train_maestro[target_name] # Aísla el vector con la variable objetivo clínica a predecir (Y)
    
    # Determinar si es binario o multiclase
    clases_unicas = np.unique(y_train) # Extrae los valores únicos que toma la variable objetivo (ej. 0 y 1)
    is_multiclass = len(clases_unicas) > 2 # Verifica mediante una expresión booleana si hay más de 2 categorías

    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") # Matriz para control

    # 4. Configurar Grid Search (5 Pliegues) CON PIPELINE Y SEMILLA
    # Etapa 3: Configuración del proceso de optimización de hiperparámetros con validación cruzada estratificada
    print("[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...") 
    # Instancia el particionador de 5 pliegues garantizando los mismos cortes de validación
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Para multiclase en LR nativo de sklearn usando 'saga', multinomial es ideal estadísticamente
    multi_class_strat = 'multinomial' if is_multiclass else 'auto' # Selecciona la estrategia de entropía cruzada multiclase si corresponde

    pipeline = Pipeline([ # Construye el tubo de ejecución para evitar fugas de información
        ('scaler', StandardScaler()), # Paso 1: Escalar características numéricas para que la regularización funcione bien
        ('lr', LogisticRegression(solver='saga', max_iter=1000, class_weight='balanced',  # Paso 2: Configura el motor de la regresión
                                  multi_class=multi_class_strat, random_state=42, n_jobs=1)) # Mantiene n_jobs=1 aquí para no chocar con el GridSearchCV
    ])

    espacio_hiperparametros = { # Define las combinaciones exactas exigidas en el documento de propuesta
        'lr__penalty': ['l1', 'l2'], # Configura la regularización fuerte (Lasso) y suave (Ridge)
        'lr__C': [0.1, 1.0, 10.0] # Configura el inverso de la fuerza de regularización
    }

    grid_search = GridSearchCV( # Instancia la validación cruzada exhaustiva
        estimator=pipeline, # Asigna el pipeline que primero escala y luego predice
        param_grid=espacio_hiperparametros, # Asigna el diccionario de hiperparámetros (6 combinaciones en total)
        cv=cv_estrategia, # Asigna los 5 pliegues estrictamente sembrados
        scoring='f1_macro', # Establece F1-Macro como la métrica reina a maximizar
        n_jobs=-1, # Usa todos los hilos del PC para evaluar pliegues en paralelo
        verbose=3 # Activa el nivel 3 de mensajes para mostrar por consola el progreso de cada iteración y su tiempo
    )

    # 5. Entrenar y extraer métricas filtradas
    # Etapa 4: Ejecución del proceso de entrenamiento y optimización
    print("[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...") 
    inicio = time.time() # Toma una foto del tiempo del sistema antes de empezar
    grid_search.fit(X_train, y_train) # Desencadena el proceso: 6 configuraciones x 5 pliegues = 30 ajustes totales
    fin = time.time() # Toma una foto del tiempo del sistema al concluir
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") # Calcula e imprime la duración del entrenamiento

    # -------------------------------------------------------------------------
    # Extracción de resultados, generación de respaldo CSV y filtrado metodológico
    # -------------------------------------------------------------------------
    cv_resultados = pd.DataFrame(grid_search.cv_results_) # Transforma el histórico interno de resultados en un DataFrame manejable
    
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_LR_{target_name}.csv") # Construye la ruta final para guardar el registro CSV
    cv_resultados.to_csv(ruta_cv, index=False) # Escribe el registro físico ignorando el índice automático numérico
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}") # Imprime la confirmación de la auditoría técnica

    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] # Filtra metodológicamente descartando resultados con alta varianza

    if config_estables.empty: # Condición para verificar si no sobrevivió ningún hiperparámetro al filtro de estabilidad
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") # Levanta una alarma al usuario en la terminal
        print("      Se utilizará la de mayor promedio por defecto.") # Informa la adopción de la política por defecto de Scikit-Learn
        mejor_modelo = grid_search.best_estimator_ # Captura directamente el modelo original sin filtro
    else: # Si existen configuraciones que pasaron la validación de robustez
        mejor_indice = config_estables['mean_test_score'].idxmax() # Identifica qué fila del DataFrame filtrado tiene el F1-Score promedio más alto
        mejores_params = config_estables.loc[mejor_indice, 'params'] # Extrae como diccionario la combinación de esa fila
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] # Recupera el valor de la métrica media obtenida
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] # Recupera el valor de la desviación estándar (inferior a 0.10)
        
        print(f"      -> Mejor configuración estable encontrada:") # Anuncia la extracción existosa
        print(f"         Hiperparámetros: {mejores_params}") # Lista el diccionario de parámetros seleccionados
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") # Imprime sus números para justificación científica

        mejor_modelo = clone(grid_search.estimator) # Fabrica una copia en limpio del pipeline arquitectónico sin memoria de datos
        mejor_modelo.set_params(**mejores_params) # Le inyecta la combinación exacta aprobada por el filtro
        mejor_modelo.fit(X_train, y_train) # Reentrena la estructura vacía usando la totalidad de los datos para producción
        
    # -------------------------------------------------------------------------

    del df_train_maestro, X_train, y_train # Liquida los vectores de entrenamiento de la memoria RAM
    gc.collect() # Fuerza la reubicación de memoria del sistema operativo

    # 6. Evaluación en el Conjunto de Prueba
    # Etapa 5: Testeo de capacidad de generalización del modelo sobre el futuro (dataset del año 2024)
    print("[5/5] Evaluando en conjunto de prueba (2024)...") # Inicia el testeo de capacidad de generalización sobre el futuro
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), low_memory=False) # Absorbe las admisiones con cáncer del 2024
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), low_memory=False) # Absorbe los controles correspondientes del 2024

    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) # Alinea ambas tablas para reconstruir el grupo de prueba íntegro
    X_test = df_test_maestro[features] # Extrae los atributos predictores enmascarados de los pacientes nuevos
    y_test = df_test_maestro[target_name] # Toma la verdad empírica (diagnóstico u ocurrencia real)

    y_pred = mejor_modelo.predict(X_test) # Ejecuta el veredicto clínico duro del modelo (clase exacta asignada)
    y_pred_proba = mejor_modelo.predict_proba(X_test) # Pide el cálculo matemático blando que respalda la decisión anterior (probabilidad)

    print("\n--- RESULTADOS FINALES ---") 
    print(classification_report(y_test, y_pred)) # Muestra desglose exhaustivo de las tasas de precisión y recall de la librería nativa
    
    f1_macro_val = f1_score(y_test, y_pred, average='macro') # Consolida el balance F1 ponderándolo de forma no viciada entre clases
    
    if is_multiclass: # Control de flujo lógico para bifurcar si el objetivo tiene más de 2 categorías (Severidad, Consumo)
        y_test_bin = label_binarize(y_test, classes=clases_unicas) # Expande la etiqueta real en un vector multidimensional One-vs-Rest (Ej: 2 -> [0, 0, 1])
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') # Mide el área discriminativa de las múltiples curvas ROC simultáneas
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') # Mide el área de la curva de precisión multiclase
        
        # Iterador interno por compresión de listas que calcula el BS por cada clase individual y luego los promedia
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))])
        
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Imprime la métrica cardinal F1 general
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") # Imprime el AUPRC ajustado por peso de clase
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") # Imprime el ROC multiclase
        print(f"Brier Score (Multiclase): {brier_val:.4f}") # Imprime la penalización de Brier promediada
        
        # 7. Extraer Odds Ratios por clase
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ # Rescata los valores numéricos con los que StandardScaler encogió los datos originales
        for idx, clase in enumerate(clases_unicas): # Inicia un bucle que procesa cada una de las clases de forma independiente
            coef_lr = mejor_modelo.named_steps['lr'].coef_[idx] # Captura el vector de pesos algebraicos subyacente para la clase analizada
            coef_orig = coef_lr / desviaciones_std # Reprocesa el peso algebraico multiplicándolo en reverso para volverlo a la escala humana
            
            df_coef = pd.DataFrame({ # Inicializa una tabla que cruzará la variable con su nuevo peso natural
                'Variable': features, # Escribe en la primera columna el listado textual de atributos
                'Coeficiente_Original': coef_orig, # Añade en la segunda columna el peso logarítmico reconstituido
                'Odds_Ratio': np.exp(coef_orig) # Efectúa la función exponencial de euler para devolver la interpretación "Odds Ratio" clínica
            }).sort_values(by='Odds_Ratio', ascending=False) # Fuerza un ordenamiento para que la tabla comience con el de mayor probabilidad
            
            ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}_Clase_{clase}.csv") # Dinamiza el nombre del CSV dependiendo de la categoría evaluada
            df_coef.to_csv(ruta_coef, index=False) # Exporta silenciosamente el listado
            
    else: # Si el análisis corresponde únicamente al enfoque dual: Fallecido / No Fallecido
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) # Obliga a medir el F1 tomando como prioridad exclusivamente el caso minoritario
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) # Evalúa la capacidad general del modelo separando el eje vivo/muerto
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) # Examina rigurosamente qué tan preciso se es a medida que se aumenta el recall de fallecidos
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) # Cuantifica directamente el error cuadrado medio entre la probabilidad otorgada y si el sujeto falleció
        
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") # Muestra F1 de la clase fallecida
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Muestra el equilibrio armónico bidireccional
        print(f"AUPRC: {auprc_val:.4f}") # Exhibe AUPRC
        print(f"AUC-ROC: {auc_roc_val:.4f}") # Exhibe AUC-ROC
        print(f"Brier Score: {brier_val:.4f}") # Exhibe el calibrador probabilístico

        # 7. Extraer Odds Ratios Binario
        coef_lr = mejor_modelo.named_steps['lr'].coef_[0] # Toma el vector de pesos algebraicos crudos que dictaminan la mortalidad
        desviaciones_std = mejor_modelo.named_steps['scaler'].scale_ # Toma la deformación impuesta por StandardScaler
        coef_orig = coef_lr / desviaciones_std # Deshace aritméticamente el encogimiento para anular la distorsión del pipeline
        
        df_coef = pd.DataFrame({ # Empaqueta el cruce de dimensiones en una estructura tabular simple
            'Variable': features, # Inserta los literales referenciales
            'Coeficiente_Original': coef_orig, # Inserta las razones logarítmicas base
            'Odds_Ratio': np.exp(coef_orig) # Inserta los multiplicadores de riesgo (Odds Ratio)
        }).sort_values(by='Odds_Ratio', ascending=False) # Mueve hacia arriba la variable más letal
        
        ruta_coef = os.path.join(dir_resultados, f"LR_OddsRatios_{target_name}.csv") # Genera el nombre del archivo de salida
        df_coef.to_csv(ruta_coef, index=False) # Completa la exportación física 

    print(f"Odds Ratios guardados en: {dir_resultados}") # Aviso de culminación exitosa
    
    del df_test_maestro, X_test, y_test # Libera las matrices del año 2024 de la memoria primaria
    gc.collect() # Fuerz el vaciado explícito
    print("="*60, "\n")

In [2]:
entrenar_evaluar_lr('MORTALIDAD')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


      -> Búsqueda completada en 106.28 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_MORTALIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 10.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.6035 (std: 0.0006)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (2024)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       1.00      0.86      0.92   1050124
           1       0.14      0.90      0.24     26221

    accuracy                           0.86   1076345
   macro avg       0.57      0.88      0.58   1076345
weighted avg       0.98      0.86      0.91   1076345

F1-Score (Clase 1): 0.2362
F1-Score (Macro): 0.5791
AUPRC: 0.2785
AUC-ROC: 0.9424
Brier Score: 0.0967
Odds Ratios guardados en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica



In [3]:
entrenar_evaluar_lr('SEVERIDAD')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...


[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> Búsqueda completada en 118.71 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_SEVERIDAD.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 10.0, 'lr__penalty': 'l1'}
         F1-Macro Promedio: 0.7113 (std: 0.0012)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (2024)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       0.89      0.97      0.93    211972
           1       0.70      0.76      0.73    383281
           2       0.52      0.36      0.43    266701
           3       0.63      0.72      0.67    214391

    accuracy                           0.69   1076345
   macro avg       0.69      0.70      0.69   1076345
weighted avg       0.68      0.69      0.68   1076345

F1-Score (Macro): 0.6889
AUPRC (OvR Weighted): 0.7470
AUC-ROC (OvR Weighted): 0.8871
Brier Score (Multiclase): 0.0990
Odds Ratios guardados en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica



In [4]:
entrenar_evaluar_lr('CONSUMO_RECURSOS')

INICIANDO PIPELINE DE REGRESIÓN LOGÍSTICA PARA: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con Pipeline y semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


      -> Búsqueda completada en 87.74 minutos.
      -> Evidencia de hiperparámetros guardada en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica\Resultados_GridSearch_LR_CONSUMO_RECURSOS.csv
      -> Mejor configuración estable encontrada:
         Hiperparámetros: {'lr__C': 0.1, 'lr__penalty': 'l2'}
         F1-Macro Promedio: 0.6687 (std: 0.0005)


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[5/5] Evaluando en conjunto de prueba (2024)...

--- RESULTADOS FINALES ---
              precision    recall  f1-score   support

           0       0.69      0.83      0.75    360074
           1       0.60      0.59      0.60    362459
           2       0.75      0.60      0.67    353812

    accuracy                           0.67   1076345
   macro avg       0.68      0.67      0.67   1076345
weighted avg       0.68      0.67      0.67   1076345

F1-Score (Macro): 0.6714
AUPRC (OvR Weighted): 0.7400
AUC-ROC (OvR Weighted): 0.8497
Brier Score (Multiclase): 0.1439
Odds Ratios guardados en: ../../Resultados/Resultados (etapa 3)/Regresion_Logistica



In [ ]:

# Para ejecutarlo, simplemente llamas a la función con el nombre de tus variables:
# entrenar_evaluar_lr('MORTALIDAD')
# entrenar_evaluar_lr('SEVERIDAD')
# entrenar_evaluar_lr('CONSUMO_RECURSOS')

AUPRC (Área bajo la curva Precision-Recall): Evalúa la capacidad del modelo para detectar las clases de interés. Para variables con desbalance severo (como la mortalidad, con prevalencia $\approx$ 2.4%), se exigió superar en al menos 3 veces la tasa base (Lift $>$ 3.0). Para las variables ordinales multiclase (Severidad y Consumo de Recursos), cuyas prevalencias basales oscilan entre el 25% y el 33%, la exigencia matemática de un Lift $>$ 3.0 resulta asintótica o superior a 1.0. Por tanto, para estos escenarios se reportará el promedio ponderado (One-vs-Rest), considerándose exitoso un modelo que supere sustancialmente el desempeño dictado por el azar (Lift $>$ 2.0).

In [6]:
import pandas as pd # Importa Pandas para leer los archivos CSV
import numpy as np # Importa Numpy para cálculos matemáticos
import os # Importa OS para manejar las rutas de los archivos

def validar_auprc_lift_automatico(target_name, auprc_obtenido, is_multiclass=False):
    """
    - Descripción: Lee automáticamente el conjunto de prueba (2024), calcula la tasa base (prevalencia)
                   directamente de los datos y verifica matemáticamente si el AUPRC cumple con Lift > 3.0.
    - Entrada: target_name (str) - Nombre del objetivo, auprc_obtenido (float) - El AUPRC de tu mejor modelo,
               is_multiclass (bool) - True si tiene más de 2 clases.
    - Salida: Imprime el reporte de validación en la consola.
    """
    print("="*60) # Separador visual
    print(f"VALIDACIÓN AUTOMÁTICA DE AUPRC: {target_name.upper()}") # Título
    print("="*60) # Separador visual
    
    # 1. Leer SOLO la columna objetivo de los archivos de 2024 (es ultrarrápido y no gasta RAM)
    dir_datos = "../../Datos/Datasets Finales" # Define la ruta relativa
    df_onco = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), usecols=[target_name]) # Carga onco
    df_control = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), usecols=[target_name]) # Carga control
    
    # 2. Unir y contar las instancias reales
    y_test = pd.concat([df_onco, df_control], ignore_index=True)[target_name] # Une ambos vectores
    clases, soportes_clases = np.unique(y_test, return_counts=True) # Extrae las clases y cuenta cuántas hay de cada una
    total_instancias = len(y_test) # Obtiene el total de pacientes evaluados
    
    # 3. Calcular la Tasa Base según el tipo de problema
    if not is_multiclass: # Si es Mortalidad (Binario)
        indice_clase_1 = np.where(clases == 1)[0][0] # Encuentra en qué posición del arreglo está la clase 1 (Fallecidos)
        soporte_clase_positiva = soportes_clases[indice_clase_1] # Extrae la cantidad exacta de fallecidos
        tasa_base = soporte_clase_positiva / total_instancias # Calcula la prevalencia pura
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Casos reales clase minoritaria (1): {soporte_clase_positiva}") # Muestra casos positivos
        print(f"Tasa base (Prevalencia): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra porcentaje
        
    else: # Si es Severidad o Consumo (Multiclase)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases] # Calcula la prevalencia individual de cada clase
        tasa_base = sum([p**2 for p in prevalencias]) # Calcula la prevalencia esperada por azar (promedio ponderado OvR)
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Prevalencias exactas por clase: {[round(p, 4) for p in prevalencias]}") # Muestra cómo se reparte la torta
        print(f"Tasa base ponderada (OvR): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra el azar ponderado

    # 4. Calcular el umbral y el Lift real
    umbral_minimo = tasa_base * 3.0 # Calcula el AUPRC mínimo que exige tu tesis
    lift_real = auprc_obtenido / tasa_base # Calcula cuántas veces mejor que el azar es tu modelo
    
    print("-" * 60) # Separador
    print(f"- AUPRC Obtenido por tu modelo: {auprc_obtenido:.4f}") # Muestra tu resultado
    print(f"- AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}") # Muestra el límite inferior
    print(f"- LIFT REAL LOGRADO: {lift_real:.2f}x") # Muestra el multiplicador de mejora
    
    # 5. Veredicto final
    if auprc_obtenido > umbral_minimo: # Si supera el umbral
        print("RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)")
    else: # Si no lo supera
        print("RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)")
    print("\n")

In [8]:
# 1. MORTALIDAD (Mejor Modelo: Random Forest -> AUPRC = 0.3561)
validar_auprc_lift_automatico(
    target_name='MORTALIDAD', 
    auprc_obtenido=0.2785, 
    is_multiclass=False
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: MORTALIDAD
Total episodios de prueba: 1076345
Casos reales clase minoritaria (1): 26221
Tasa base (Prevalencia): 0.0244 (2.44%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.2785
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.0731
- LIFT REAL LOGRADO: 11.43x
RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)




In [9]:
# 2. SEVERIDAD (Mejor Modelo: XGBoost -> AUPRC = 0.7814)
validar_auprc_lift_automatico(
    target_name='SEVERIDAD', 
    auprc_obtenido=0.747, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: SEVERIDAD
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [0.1969, 0.3561, 0.2478, 0.1992]
Tasa base ponderada (OvR): 0.2667 (26.67%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.7470
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.8000
- LIFT REAL LOGRADO: 2.80x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)




In [10]:
# 3. CONSUMO DE RECURSOS (Mejor Modelo: XGBoost -> AUPRC = 0.8205)
validar_auprc_lift_automatico(
    target_name='CONSUMO_RECURSOS', 
    auprc_obtenido=0.74, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: CONSUMO_RECURSOS
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [0.3345, 0.3367, 0.3287]
Tasa base ponderada (OvR): 0.3334 (33.34%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.7400
- AUPRC Mínimo exigido (Tasa Base x 3.0): 1.0001
- LIFT REAL LOGRADO: 2.22x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)


